In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from collections import Counter
from pathlib import Path

In [ ]:
base_dir = Path.cwd()
OUTPUT_DIR = base_dir / "output"
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-dnam")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.dnam.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

RANDOM_SEED = 42
N_JOBS = 4

FEATURE_SELECTION_PERCENTILE = 10  # KEEP TOP n% OF CPG SITES
N_SPLITS = 5

from helpers import run_cv, analyze_feature_importances

In [3]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train = X_train.astype(np.float32)

y_train = pd.read_csv(y_train_path, index_col=0)

X_train, y_train

(              cg00000029  cg00000109  cg00000165  cg00000236  cg00000289  \
 id                                                                         
 ALL_1277          0.7557      0.9169      0.0990      0.9222      0.7708   
 ALL_1286          0.8646      0.8970      0.1051      0.9038      0.6404   
 ALL_1300          0.8852      0.8994      0.0783      0.8878      0.6192   
 ALL_1322          0.7903      0.9051      0.2426      0.9112      0.6721   
 ALL_1352          0.9116      0.8630      0.1516      0.8653      0.7713   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2991      0.3880      0.9099      0.1818      0.5858      0.8566   
 TCGA-AB-2992      0.9088      0.9207      0.1163      0.8856      0.7777   
 TCGA-AB-2997      0.2163      0.9154      0.1116      0.4988      0.8602   
 TCGA-AB-3007      0.7086      0.8974      0.1788      0.5349      0.8395   
 TCGA-AB-3012      0.5843      0.9399      0.1827      0.6158      0.8592   

In [4]:
# Show class distributions in y_train and y_test
print("y_train class distribution:")
print(y_train.value_counts())

y_train class distribution:
class                   
AML, KMT2A-r                241
B-ALL, high hyperdiploid    229
B-ALL, ETV6::RUNX1          199
AML, RUNX1::RUNX1T1         174
AML, NPM1alt                130
AML, CBFB::MYH11            127
Non-leukemic                 95
T-ALL                        73
AML, NUP98-r                 69
AML, CEBPAalt                55
B-ALL, PAX5alt               47
B-ALL, KMT2A-r               43
B-ALL, TCF3::PBX1            26
B-ALL, DUX4-r                22
AML, CBFA2T3::GLIS2          22
AML, DEK::NUP214             19
B-ALL, BCR::ABL1             17
AML, APL-PML::RARA           16
AML, ETV6-r                  16
B-ALL, ETV6::RUNX1-like      15
B-ALL, iAMP21                15
B-ALL, ZNF384-r              15
B-ALL, BCR::ABL1-like        14
AML, KAT6A-r                  8
AML, PICALM::MLLT10           6
AML, FUS::ERG                 6
B-ALL, MEF2D-r                6
B-ALL, PAX5 P80R              6
Name: count, dtype: int64


In [5]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_train

array(['B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', ..., 'AML, APL-PML::RARA',
       'AML, APL-PML::RARA', 'AML, APL-PML::RARA'],
      shape=(1711,), dtype=object)

In [ ]:
cv_results = run_cv(X_train, y_train)
importance_df, freq_df, selected_features = analyze_feature_importances(OUTPUT_DIR,cv_results, X_train, suffix="1", freq_threshold=1)
X_reduced = X_train[selected_features]
X_reduced

In [57]:
cv_results = run_cv(X_reduced, y_train)
cv_results


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


[CV] END .................................................... total time=48.3min
[CV] END .................................................... total time=48.7min
[CV] END .................................................... total time=49.8min
[CV] END .................................................... total time=51.0min
[CV] END .................................................... total time=24.9min
Train F1-macro: 1.000 ± 0.000
Val   F1-macro: 0.817 ± 0.029
Gap:            0.183


[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed: 73.3min finished


{'fit_time': array([2919.43718767, 2899.88704824, 2987.52287769, 3057.48615742,
        1493.63082409]),
 'score_time': array([0.54606557, 0.72213411, 0.70561314, 0.49756241, 0.548311  ]),
 'estimator': [RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42),
  RandomForestClassifier(class_weight='balanced', max_depth=40, max_features=0.25,
                         n_jobs=1, random_state=42)],
 'test_score': array([0.81904817, 0.8337073 , 0.85686333, 0.76903327, 0.80813924]),
 'train_score': array([1., 1., 1.

In [ ]:
importance_df, freq_df, selected_features = analyze_feature_importances(OUTPUT_DIR, cv_results, X_reduced, suffix="2", freq_threshold=1)

Selected 17343 features with frequency >= 1. Saved to /home/mariya/Development/pallas/training/dnam/output/selected_features.2.txt.
